In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

In [ ]:
df = pd.read_csv('titanic_data_updated.csv')
df

### splitting data

In [ ]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

# family_size creation
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

#feature and target extract
X = df.drop(['Survived'],axis=1)
y = df['Survived']


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

### imputation of data

In [ ]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan , strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan , strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan , strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)
imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(X_train)

X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)

### outlier handling of age

In [ ]:
mean_of_age = X_train['Age'].mean()
std_of_age = X_train['Age'].std()
X_train['zscore_Age'] = (X_train['Age']-mean_of_age)/std_of_age

X_train = X_train[abs(X_train['zscore_Age']) <=3]
X_train.drop(['zscore_Age'],axis=1 , inplace=True)

### Outlier handling of Fare

In [ ]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
fare_IQR = fare_Q3 - fare_Q1
fare_minimum = max(0,fare_Q1 - 1.5 * fare_IQR)
fare_maximum = fare_Q3 + 1.5 * fare_IQR

X_train['Fare']= X_train['Fare'].clip(fare_minimum , fare_maximum)

### encoding and scaling

In [ ]:
X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck'] = X_test['Cabin'].astype(str).str[0]

In [ ]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('embarked_sex',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex','Cabin_Deck']),
        ('age_scaler',StandardScaler(),['Age']),
        ('fare_scaler',MinMaxScaler(),['Fare','FamilySize'])
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)
encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(X_train)

X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

In [ ]:
X_train.drop(['Cabin','SibSp','Parch'],axis=1,inplace=True)
X_test.drop(['Cabin','SibSp','Parch'],axis=1,inplace=True)

### Final Dataset

In [ ]:
X_train

In [ ]:
X_test